# RAG from scratch

Pipeline: **load** → chunk → embed → store → retrieve → generate

## Step 1: Load
Goal: turn each PDF into plain text, and keep track of **where the text came from** (file + page).
That "where" is called **metadata**. We need it later to cite sources and to check retrieval against `eval/questions.json`.

In [4]:
from pathlib import Path
from pypdf import PdfReader

DATA_DIR = Path("../data/raw")  # notebooks run fraom the notebooks/ folder

def load_pdfs(folder):
    """Return one record per PDF page: {text, source, page}."""
    pages = []
    for pdf_path in sorted(folder.glob("*.pdf")):
        reader = PdfReader(pdf_path)
        for page_num, page in enumerate(reader.pages, start=1):
            text = page.extract_text() or ""  # scanned/image pages return None
            pages.append({"text": text, "source": pdf_path.name, "page": page_num})
    return pages

pages = load_pdfs(DATA_DIR)
print(f"Loaded {len(pages)} pages")
for p in pages:
    print(f"{p['source']:40} page {p['page']}  {len(p['text']):5} chars")

Loaded 9 pages
01_caching_and_cdn.pdf                   page 1   2475 chars
01_caching_and_cdn.pdf                   page 2   2956 chars
01_caching_and_cdn.pdf                   page 3   1811 chars
02_project_falcon_databases.pdf          page 1   2578 chars
02_project_falcon_databases.pdf          page 2   2761 chars
02_project_falcon_databases.pdf          page 3   1322 chars
03_shopkart_checkout_case_study.pdf      page 1   2704 chars
03_shopkart_checkout_case_study.pdf      page 2   2308 chars
03_shopkart_checkout_case_study.pdf      page 3    306 chars


### Inspect the raw text
Always look at what the loader produced. Check: did the **table** survive? Are there broken words, extra line breaks, or headers/footers repeated on every page?

In [5]:
print(pages[0]["text"][:3000])

Caching and CDN Design Guide
Halcyon Labs Platform Engineering · Internal standard PE-CACHE-07 · Revision 3
1. Why we cache
Caching is the practice of storing the result of an expensive operation so that later requests can be
served faster and more cheaply. At Halcyon Labs almost every read path passes through at least
one cache layer before it touches a database. A well-designed cache reduces latency, protects
downstream systems from load spikes and lowers infrastructure cost. A badly designed cache
serves stale data, hides bugs and fails in surprising ways during incidents.
This guide explains the general principles first and then describes the specific defaults that Halcyon
Labs teams must follow. Where this document says must, a deviation needs approval from the
Platform Architecture Review, which meets every second Thursday.
2. Cache placement and tiers
Caches can live in several places: in the client (browser or mobile app), at the edge (a CDN), in a
shared distributed cache such

## Step 2: Chunk
Goal: split each page into small pieces (**chunks**) that each hold one idea.

Why not embed a whole page?
- One vector for a whole page blurs many topics together, so search gets less precise.
- The LLM prompt gets filled with irrelevant text.
- Embedding models have a max input (bge-small: 512 tokens); longer text gets cut off.

Strategy here: **fixed-size word chunks with overlap** (the simplest one).
- `chunk_size=150` words (~200 tokens). Our docs are small, so small chunks make retrieval actually work.
- `overlap=30` words: neighbouring chunks share some words, so a sentence cut at a boundary still appears whole in one of them.

In [6]:
def chunk_text(text, chunk_size=150, overlap=30):
    """Split text into chunks of `chunk_size` words; neighbours share `overlap` words."""
    words = text.split()            # split on any whitespace -> list of words
    step = chunk_size - overlap     # how far to move forward each time (120)
    chunks = []
    for start in range(0, len(words), step):   # start = 0, 120, 240, ...
        chunks.append(" ".join(words[start:start + chunk_size]))
        if start + chunk_size >= len(words):   # this chunk reached the end
            break
    return chunks

# Try it on a tiny example first: 10 "words", chunks of 4, overlap 1
print(chunk_text("w1 w2 w3 w4 w5 w6 w7 w8 w9 w10", chunk_size=4, overlap=1))

['w1 w2 w3 w4', 'w4 w5 w6 w7', 'w7 w8 w9 w10']


In [7]:
chunks = []
for p in pages:
    for i, text in enumerate(chunk_text(p["text"])):
        chunks.append({
            "id": f"{p['source']}-p{p['page']}-c{i}",  # unique ID, e.g. 01_caching_and_cdn.pdf-p1-c0
            "text": text,
            "source": p["source"],  # metadata copied from the page
            "page": p["page"],
        })

print(f"{len(chunks)} chunks")
print("words per chunk:", [len(c["text"].split()) for c in chunks])

30 chunks
words per chunk: [150, 150, 150, 55, 150, 150, 150, 133, 150, 150, 55, 150, 150, 150, 35, 150, 150, 150, 101, 150, 96, 150, 150, 150, 71, 150, 150, 150, 31, 50]


In [8]:
# Check the overlap: the end of chunk 0 should repeat at the start of chunk 1
print("END OF CHUNK 0:  ...", chunks[0]["text"][-150:])
print()
print("START OF CHUNK 1:", chunks[1]["text"][:150], "...")

END OF CHUNK 0:  ... e Review, which meets every second Thursday. 2. Cache placement and tiers Caches can live in several places: in the client (browser or mobile app), at

START OF CHUNK 1: approval from the Platform Architecture Review, which meets every second Thursday. 2. Cache placement and tiers Caches can live in several places: in  ...


## Step 3: Embed
Goal: turn every chunk into a **vector** (384 numbers) so we can compare meanings with math.

- Model: `BAAI/bge-small-en-v1.5`, runs locally on CPU, free.
- `normalize_embeddings=True` makes every vector length 1, so **dot product = cosine similarity**. That makes search in step 4 a single matrix multiplication.
- This is the **indexing** side of RAG: done once, offline. Only the question gets embedded at query time.

In [9]:
from sentence_transformers import SentenceTransformer
import numpy as np

# First run downloads the model (~130 MB) into a cache; later runs load it from disk
model = SentenceTransformer("BAAI/bge-small-en-v1.5")
print("vector size:", model.get_embedding_dimension())

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

vector size: 384


In [10]:
# Feel what similarity means before embedding the chunks
sentences = [
    "The cache TTL is 47 seconds",            # A
    "Cached items expire after 47 seconds",   # B: same meaning, different words
    "ShopKart holds inventory for 7 minutes", # C: different topic
]
vecs = model.encode(sentences, normalize_embeddings=True)
print("A vs B (same meaning):", round(float(vecs[0] @ vecs[1]), 3))
print("A vs C (different):   ", round(float(vecs[0] @ vecs[2]), 3))

A vs B (same meaning): 0.809
A vs C (different):    0.651


In [11]:
texts = [c["text"] for c in chunks]            # just the text of every chunk
embeddings = model.encode(texts, normalize_embeddings=True, show_progress_bar=True)

print("shape:", embeddings.shape)                # (number of chunks, 384)
print("first 5 numbers of chunk 0:", embeddings[0][:5])

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

shape: (30, 384)
first 5 numbers of chunk 0: [-0.06061042  0.0228219   0.01824471  0.00942019 -0.00876194]


## Step 4: Store + Retrieve
**Store:** our "vector DB" is just the `embeddings` array (30 × 384) in memory, plus `chunks` for the text. Row `i` of `embeddings` ↔ `chunks[i]`.

**Retrieve (brute-force search):**
1. Embed the question with the **same model**.
2. Score it against every chunk in one go: `embeddings @ q_vec` → 30 scores.
3. Sort and take the top **k** (k = 5).

In [12]:
def search(question, k=5):
    """Return the k chunks most similar to the question (brute-force search)."""
    q_vec = model.encode(question, normalize_embeddings=True)  # shape (384,)
    scores = embeddings @ q_vec               # shape (30,): one similarity score per chunk
    top_idx = np.argsort(scores)[::-1][:k]    # indexes of the k highest scores
    return [{**chunks[i], "score": float(scores[i])} for i in top_idx]

In [13]:
def show(results):
    for r in results:
        print(f"{r['score']:.3f}  {r['id']}")
        print("       ", r["text"][:120], "...")

show(search("What is the default TTL for Halcyon's shared Redis cache?"))

0.803  01_caching_and_cdn.pdf-p1-c1
        approval from the Platform Architecture Review, which meets every second Thursday. 2. Cache placement and tiers Caches c ...
0.740  01_caching_and_cdn.pdf-p1-c2
        (local) 30 seconds Browser managed T1 – Edge CDN (provider: Stratus Edge) 8–20 ms 10 minutes for static, 0 for HTML Unli ...
0.715  01_caching_and_cdn.pdf-p1-c0
        Caching and CDN Design Guide Halcyon Labs Platform Engineering · Internal standard PE-CACHE-07 · Revision 3 1. Why we ca ...
0.697  01_caching_and_cdn.pdf-p2-c1
        for any data involving money, after the incident described in section 6.  Refresh-ahead: the cache proactively reloads  ...
0.689  02_project_falcon_databases.pdf-p1-c0
        Project Falcon: Database Scaling Design Halcyon Labs Data Infrastructure · Design document DI-FALCON-002 · Approved Marc ...


In [14]:
# A question that is NOT in the documents (q24 in eval/questions.json)
show(search("How many virtual nodes does ShopKart's consistent hashing ring use?"))

0.815  02_project_falcon_databases.pdf-p2-c2
        of nearly all of them, as happens with modulo hashing. With only a few nodes, the ring is uneven: one node might own a m ...
0.777  02_project_falcon_databases.pdf-p2-c1
        cross-shard joins. The team considered order_id as the shard key but rejected it because merchant dashboards would then  ...
0.742  02_project_falcon_databases.pdf-p2-c0
        Strategy How it works Strength Weakness Used in Falcon? Hash-based (modulo) shard = hash(key) mod N Even distribution Ad ...
0.731  03_shopkart_checkout_case_study.pdf-p2-c0
         A token bucket limiter at the API gateway allows each user 5 checkout requests per second with a burst of 10.  A glob ...
0.691  03_shopkart_checkout_case_study.pdf-p1-c0
        Case Study: How ShopKart Scaled Checkout Engineering case study · ShopKart (fictional Indian e-commerce company) · Mega  ...


## Step 5: Generate
Goal: give the LLM the question + the retrieved chunks, and let it write the answer.

The prompt has 3 parts:
1. **Instructions:** answer only from the context, say "I don't know" otherwise, cite sources. This is the safety net for the step 4 trap (search always returns *something*).
2. **Context:** the top-k chunks, each labelled with its source and page.
3. **Question**

This is the **A** in RAG: **augmenting** the prompt with retrieved text. Needs Ollama running (check the tray icon).

In [16]:
PROMPT_TEMPLATE = """You are a helpful assistant. Answer the question using ONLY the context below.
If the answer is not in the context, reply exactly: "I don't know based on the documents."
Cite the source of each fact like [file, page N].

Context:
{context}

Question: {question}
Answer:"""

def build_prompt(question, results):
    # Label each chunk with its source so the LLM can cite it
    context = "\n\n".join(f"[{r['source']}, page {r['page']}]\n{r['text']}" for r in results)
    return PROMPT_TEMPLATE.format(context=context, question=question)

# Look at the exact prompt the LLM will receive
print(build_prompt("What is the default TTL for Halcyon's shared Redis cache?",
                   search("What is the default TTL for Halcyon's shared Redis cache?", k=2)))

You are a helpful assistant. Answer the question using ONLY the context below.
If the answer is not in the context, reply exactly: "I don't know based on the documents."
Cite the source of each fact like [file, page N].

Context:
[01_caching_and_cdn.pdf, page 1]
approval from the Platform Architecture Review, which meets every second Thursday. 2. Cache placement and tiers Caches can live in several places: in the client (browser or mobile app), at the edge (a CDN), in a shared distributed cache such as Redis, or inside the application process itself. Each tier trades freshness for speed. The closer a cache is to the user, the faster it is, and the harder it is to invalidate. Halcyon Labs uses four tiers. The table below lists the standard configuration for each one. Teams may lower a TTL without approval, but raising it above the listed value requires review. Tier Technology Typical latency Default TTL Size limit T0 – Client HTTP Cache-Control headers 0 ms (local) 30 seconds Browser ma

In [17]:
import ollama

def answer(question, k=5):
    """The full RAG pipeline: retrieve the top-k chunks, then let the LLM answer from them."""
    results = search(question, k)                    # R: retrieve (step 4)
    prompt = build_prompt(question, results)         # A: augment the prompt with the chunks
    response = ollama.chat(                          # G: generate
        model="llama3.2",
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0},                  # 0 = least random, most repeatable answers
    )
    return response["message"]["content"]

In [18]:
print(answer("What is the default TTL for Halcyon's shared Redis cache?"))

The default TTL for Halcyon's shared Redis cache is 47 seconds. [01_caching_and_cdn.pdf, page 1]


In [ ]:
# The traps: these answers are NOT in the documents (q23, q24 in eval/questions.json)
print(answer("How many virtual nodes does ShopKart's consistent hashing ring use?"))
print(answer("What TTL does ShopKart use for its product catalogue cache?"))